<a href="https://colab.research.google.com/github/ldfha/RotemAI/blob/main/projects/pro13cnn/cnn3subclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# MNIST dataset으로 CNN 실습
import tensorflow as tf
print(tf.__version__)
import numpy as np
import matplotlib.pyplot as plt

# 1) 데이터 준비
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(x_train[0])
print(x_train.shape)    # (60000, 28, 28)

# 채널(channel) 추가 (흑백인 경우 1)
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float') / 255.0  # 정규화
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float') / 255.0  # 정규화
print(x_train.shape)    # (60000, 28, 28, 1)

2.20.0
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   3  18  18  18 126 136
  175  26 166 255 247 127   0   0   0   0]
 [  0   0   0   0   0   0   0   0  30  36  94 154 170 253 253 253 253 253
  225 172 253 242 195  64   0   0   0   0]
 [  0   0   0   0   0   0   0  49 238 253 253 253 253 253 253 253 253 251
   93  82  82  56  39   0   0   0   0   0]
 [  0 

In [6]:
# 모델 정의
# 사용자 작성 클래스, 함수, 레이어 등을 케라스 모델 저장/리로딩시 자동으로 등록해주는 데코레이터
@tf.keras.utils.register_keras_serializable(package='custom')   # 'custom', 'losses', 'activation' ...
class MyMnistCNN(tf.keras.Model):
    def __init__(self, **kwargs):
      super().__init__(**kwargs)

      # Conv Block1
      self.conv1 = tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation='relu')
      self.pool1 = tf.keras.layers.MaxPool2D(pool_size=(2, 2))

      # Conv Block2
      self.conv2 = tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu')
      self.pool2 = tf.keras.layers.MaxPool2D(pool_size=(2, 2))

      # Fully Connected Layer
      self.flat = tf.keras.layers.Flatten()

      self.d1 = tf.keras.layers.Dense(units=64, activation='relu')
      self.drop1 = tf.keras.layers.Dropout(rate=0.2)

      self.d2 = tf.keras.layers.Dense(units=32, activation='relu')
      self.drop2 = tf.keras.layers.Dropout(rate=0.2)

      self.out = tf.keras.layers.Dense(units=10, activation='softmax')

    def call(self, inputs, training=False):
      x = self.conv1(inputs)
      x = self.pool1(x)
      x = self.conv2(x)
      x = self.pool2(x)
      x = self.flat(x)
      x = self.d1(x)
      x = self.drop1(x, training=training)
      x = self.d2(x)
      x = self.drop2(x, training=training)
      x = self.out(x)
      return x

model = MyMnistCNN()
model.build(input_shape=(None, 28, 28, 1))
print(model.summary())


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'my_mnist_cnn_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "my_mnist_cnn_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [7]:
# 모델 컴파일
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    x_train, y_train, epochs=100, batch_size=128, validation_split=0.1, callbacks=[es], verbose=1
)

# 모델 평가 : 아래 둘의 평가 점수의 차이가 크면 과적합 의심
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
print(f'train_loss {train_loss:.4f}, train_acc {train_acc:.4f}')
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'test_loss {test_loss:.4f}, test_acc {test_acc:.4f}')

Epoch 1/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.8593 - loss: 0.4419 - val_accuracy: 0.9770 - val_loss: 0.0794
Epoch 2/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9617 - loss: 0.1362 - val_accuracy: 0.9820 - val_loss: 0.0644
Epoch 3/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9717 - loss: 0.1007 - val_accuracy: 0.9840 - val_loss: 0.0563
Epoch 4/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9779 - loss: 0.0771 - val_accuracy: 0.9895 - val_loss: 0.0412
Epoch 5/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9821 - loss: 0.0630 - val_accuracy: 0.9912 - val_loss: 0.0387
Epoch 6/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9849 - loss: 0.0525 - val_accuracy: 0.9905 - val_loss: 0.0345
Epoch 7/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9866 - loss: 0.0472 - val_accuracy: 0.9900 - val_loss: 0.0389
Epoch 8/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9872 - loss: 0.0449 - val_ac